# Phase 5-B3-C｜研究負責人正式授權簽核（RA-IBKR_TSM-SMA520_P5B3C-0001）

這份 Notebook **只由研究負責人 hh4832 本人執行**，用途只有一個：簽署 B3-C 研究授權。

- **不會做的事**：不執行回測、不讀市場資料或 Holdout、不建立 binding、不消耗 batch、不合併 PR。
- **授權只有在你本人合併正式授權 PR 後才生效。**

授權範圍（已核准、固定，Notebook 不接受修改）：

| 項目 | 值 |
|---|---|
| 研究目的 | `PIPELINE_ACCEPTANCE_ONLY` |
| 市場／標的 | IBKR_TSM／TSM |
| 策略 | SMA 5/20，Spec v0.1.1 `f4f60b44…5cff34` |
| 資料 | TSM DEV Snapshot（manifest `8b6fc838…`） |
| Core | `b16179a` |
| 研究期間 | 2024-05-29～2024-06-28 |
| 成本 | IBKR_PRO_FIXED 四情境 |
| 批次上限 | max_batches 1 |
| 執行者 | `colab-hh4832-exec-01`（SINGLE_EXECUTOR_ONLY） |
| Holdout | NO_ACCESS |

**有效期間**：`issued_at` 是執行簽核格（第 3 格）的實際時間（UTC），`expires_at` = issued_at + 7 天。兩者都由程式自動寫入，不使用示範日期。

**操作步驟**：
1. 在 Colab Secrets 設定 `GITHUB_TOKEN`。權限只給 `hh4832/strategy-research-agent` 的 Contents Read/Write，並允許此 Notebook 存取。
2. 使用全新 runtime，執行「Run All」。
3. 第 1 格會先完成所有人工核可並 fail fast：token 必須證明 GitHub 身分為 hh4832，且 runtime 必須是全新的。
4. 第 2 格做 Version Sync 與簽核前檢查：授權必須尚未存在，preflight 必須是 `READY_FOR_HUMAN_DECISION`。
5. 第 3 格是簽核動作。若 push 失敗，**不要重跑第 3 格**；先檢查 PR 分支。


In [ ]:
# 第 1 格：所有人工核可集中在這裡，任何一項不成立立即停止（fail fast）
from pathlib import Path
import json, os, re, subprocess, sys, tempfile, urllib.request
from datetime import datetime, timezone

REPO = "hh4832/strategy-research-agent"
BRANCH = "feature/phase5b3c-formal-authorization"
PIN = "2f94b0c023dfa15f98c4cf3d676ceaca994289e3"                      # 簽核程式碼的 commit；之後只允許 docs/ 與 notebooks/ 變動（Version Sync）
PR_URL = "https://github.com/hh4832/strategy-research-agent/pull/18"                # 正式授權 PR = approval_ref
APPROVER = "hh4832"
ID = "RA-IBKR_TSM-SMA520_P5B3C-0001"
EXECUTOR = "colab-hh4832-exec-01"
SPEC_SHA = "f4f60b447013062324b7dc10aa0a569aebbd542796bef6ccfd2e723d0f5cff34"
CORE_SHA = "b16179a7cc077c36046574769a32c367fa476cae"
MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"
PROPOSAL = f"governance/proposals/{ID}.json"
CONFIG = "configs/research_only/tsm_sma_5_20_p5b3c_2024_06_dev_18350b3adba8e28a7e5e.json"
BOOK = "governance/research_authorizations.jsonl"
WORK = Path("/content/phase5b3c_authorization_signoff")
assert sys.version_info >= (3, 11), "需要 Python 3.11 以上"
assert re.fullmatch(r"[0-9a-f]{40}", PIN) and PR_URL.startswith("https://github.com/" + REPO + "/pull/")

# (a) 全新 runtime：工作目錄不得存在
assert not WORK.exists(), f"工作路徑已存在：請 Runtime → Disconnect and delete runtime 後重新 Run All（{WORK}）"
# (b) Colab Secret：token 只在記憶體中，不印出
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "請在 Colab Secrets 設定 GITHUB_TOKEN 並開啟此 Notebook 的存取權"
req = urllib.request.Request("https://api.github.com/user",
                             headers={"Authorization": "Bearer " + TOKEN, "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(req, timeout=25) as r:
    login = json.load(r)["login"]
assert login.lower() == APPROVER, "GitHub token 所屬帳號不是研究負責人 hh4832，停止"
print("第 1 格 PASS：GitHub 身分 =", login, "；全新 runtime；未執行任何簽核")

def run(*cmd, cwd=None, env=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd[:3]} 失敗（exit {r.returncode}）：{r.stderr[-1500:]}")
    return r


In [ ]:
# 第 2 格：乾淨 clone、Version Sync、簽核前檢查（不寫任何帳本）
run("git", "clone", "--quiet", "--single-branch", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(WORK))
HEAD = run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip()
assert run("git", "merge-base", "--is-ancestor", PIN, HEAD, cwd=WORK, check=False).returncode == 0, "PIN 不在分支歷史中"
drift = [f for f in run("git", "diff", "--name-only", PIN, HEAD, cwd=WORK).stdout.split() if f]
assert all(f.startswith(("docs/", "notebooks/")) for f in drift), f"Version Sync 失敗：PIN 之後有程式／帳本變動 {drift}"
assert run("git", "status", "--porcelain", cwd=WORK).stdout.strip() == ""
run(sys.executable, "-m", "pip", "install", "--quiet", "-e", str(WORK) + "[test,research]")
sys.path.insert(0, str(WORK / "src"))
from strategy_agent.readiness import b3c
assert b3c.ROOT == WORK, b3c.ROOT
assert b3c.CANDIDATE_AUTHORIZATION_ID == ID and b3c.PROPOSED_EXECUTOR == EXECUTOR
assert b3c.formal_authorization_state()["state"] == "NOT_GRANTED", "授權已存在或帳本狀態異常，不得重複簽署"
assert not (WORK / PROPOSAL).exists() and not (WORK / CONFIG).exists()
pf = b3c.preflight()
assert pf["preparation"] == "READY_FOR_HUMAN_DECISION" and not pf["preparation_problems"], pf["preparation_problems"]
assert {g["status"] for g in pf["gates"].values()} == {"NOT_EXECUTED"} and pf["real_research_pass"] is False
before = {p: (WORK / p).read_bytes() for p in (BOOK, "registry/research_authorization_uses.jsonl",
                                               "governance/market_readiness.json", "governance/holdout_locks.jsonl")}
print("第 2 格 PASS：HEAD", HEAD[:7], "（PIN", PIN[:7], "）；授權尚未存在；preflight READY_FOR_HUMAN_DECISION")


In [ ]:
# 第 3 格：研究負責人本人的正式簽核動作（只在本次暫存 clone 中寫入，尚未 push）
from strategy_agent.governance.research_authorization_admin import approve
issued_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()   # 實際簽核時間；不使用示範日期
assert b3c.main(["propose", "--issued-at", issued_at, "--out", str(WORK / PROPOSAL)]) == 0
proposal = json.loads((WORK / PROPOSAL).read_text())
chk = b3c.check(proposal)
assert chk["ok"] is True, chk
assert proposal["strategy"]["spec_file_sha256"] == SPEC_SHA and proposal["dataset"]["producer_commit_sha"] == CORE_SHA
assert proposal["max_batches"] == 1 and proposal["executor"] == {"executor_id": EXECUTOR, "consistency": "SINGLE_EXECUTOR_ONLY"}
record = approve(WORK / BOOK, WORK / "governance/approvers.json", proposal, approved_by=APPROVER, approval_ref=PR_URL)
assert record["authorization_id"] == ID and record["approved_by"] == APPROVER
assert b3c.main(["derive-run-config", "--authorization-id", ID, "--out", str(WORK / CONFIG)]) == 0
st = b3c.formal_authorization_state()
assert st["state"] == "GRANTED_UNUSED" and st["uses"] == 0, st
assert (WORK / BOOK).read_bytes().startswith(before[BOOK]), "既有授權鏈被改動"
for p in list(before)[1:]:
    assert (WORK / p).read_bytes() == before[p], f"{p} 不得變動"
pf2 = b3c.preflight()
assert pf2["preparation"] == "AUTHORIZED_AWAITING_GITHUB_APPROVALS" and pf2["real_research_pass"] is False, pf2
assert {g["status"] for g in pf2["gates"].values()} == {"NOT_EXECUTED"}
t = run(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_phase5b3c_preparation.py",
        "tests/test_phase4a_research_only.py", cwd=WORK, check=False)
print(t.stdout[-600:])
assert t.returncode == 0, "簽核後的狀態測試未通過；不得 push"
changed = sorted(x[3:] for x in run("git", "status", "--porcelain", "--untracked-files=all", cwd=WORK).stdout.splitlines())
assert changed == sorted([BOOK, CONFIG, PROPOSAL]), f"意外檔案變更：{changed}"

run("git", "add", "--", BOOK, CONFIG, PROPOSAL, cwd=WORK)
run("git", "-c", f"user.name={APPROVER}", "-c", f"user.email={APPROVER}@users.noreply.github.com", "commit", "-q",
    "-m", f"governance: research lead approves {ID} (PIPELINE_ACCEPTANCE_ONLY, 1 batch, {EXECUTOR}, 7 days)", cwd=WORK)
with tempfile.TemporaryDirectory() as d:          # token 只經 askpass 傳給 git；remote 與輸出都不含 token
    ask = Path(d) / "askpass.sh"
    ask.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s' \"$GIT_USERNAME\";; *) printf '%s' \"$GIT_PASSWORD\";; esac\n")
    ask.chmod(0o700)
    env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT="0", GIT_USERNAME="x-access-token", GIT_PASSWORD=TOKEN)
    res = subprocess.run(["git", "push", "origin", "HEAD:" + BRANCH], cwd=WORK, env=env, text=True, capture_output=True)
    env["GIT_PASSWORD"] = ""
if res.returncode != 0:
    raise RuntimeError(f"push 未完成（exit {res.returncode}）；授權未生效。請勿重跑本格，先檢查 PR 分支")
print("簽核 commit：", run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip())
print("授權 ID：", ID, "｜executor：", EXECUTOR, "｜max_batches：1")
print("issued_at：", record["issued_at"], "｜expires_at：", record["expires_at"])
print("狀態：GRANTED_UNUSED（push 完成；PR 合併後才生效）｜batch 0/1｜Runner 未執行｜Holdout 未讀取")
print("下一步：等 CI 全綠後由你本人合併", PR_URL)


## 驗收判準

第 3 格成功後，正式授權 PR 會多出一個由 hh4832 署名的 commit，內含以下三個檔案：

- `governance/proposals/RA-IBKR_TSM-SMA520_P5B3C-0001.json`：由 `b3c propose` 產生，有效期間是簽核當下 + 7 天。
- `governance/research_authorizations.jsonl`：追加一行 APPROVE，approval_ref 為本 PR。
- `configs/research_only/tsm_sma_5_20_p5b3c_2024_06_dev_18350b3adba8e28a7e5e.json`：由 `b3c derive-run-config` 產生。

接著由你本人完成：

1. 等 CI 全綠。CI 會用 `formal_authorization_state() == GRANTED_UNUSED` 檢查帳本、提案與設定完全一致。
2. 合併 PR，授權從此刻生效。請盡快合併，因為有效期間從 issued_at 開始計算。

授權生效後，執行仍需兩則由你本人張貼、並經 Production GitHub Verifier 驗證的留言：

- `BIND_AUTHORIZATION`：確認 Request Hash。
- `APPROVE_TASK`：確認 Task ID。

沒有這兩則留言，Runner 一律拒絕執行。
